# 붓꽃(iris)의 품종 분류

In [1]:
# '꽃잎과 꽃받침의 크기' 를 기반으로 붓꽃의 품종 분류

# 데이터셋 : iris.csv   

# 컬럼
# SepalLength  (꽃받침의 길이)
# SepalWidth   (꽃받침의 폭)
# PetalLength  (꽃잎의 길이)
# PetalWidth   (꽃임이 너비)

# Name     품종명 (Species)⭐
#     "Iris-setosa", "Iris-versicolor", "Iris-virginica"  세가지 품종

# 출처 : https://www.kaggle.com/uciml/iris

In [2]:
# CSV 파일에는 약 150개의 데이터가 있는데,  
# 100개는 학습(train)을 위해 사용,   50개는 테스트(test)를 위해 사용

In [3]:
""" 
[수행 단계]

① 필요한 import 수행

② 데이터 읽어오기
   CSV -> DataFrame (변수명 df)
   기초 통계랑 확인
   
③ 입력데이터와 타겟데이터 분리
   입력데이터 →  변수명 df_data
   타겟데이터 →  변수명 target

④ 전처리 (표준화)
   스케일러 객체 변수 → 변수명 scaler

⑤ train, test 세트 분리. 
   - 8:2로 분류 
   - 클래스별로 균등하게 분류되게 하고 
  
⑥ 최적의 하이퍼 파라미터 찾기
   SVC 의 최적 하이퍼 파라미터 찾기 수행.  GridSearchCV 사용

   param_grid = {'C': [0.1, 1, 10, 100],               
              'gamma': [0.001, 0.1, 1, 10, 100], 
              'kernel': ['linear', 'rbf']}

   최적의 모델 저장 -> 변수명 clf

⑦ test 점수 확인
    
⑧ 다른 평가지표들 확인

⑨ 예측 동작 확인

⑩ 모델 & 스케일러 저장하기
   모델 -> 파일명 iris_model.pkl
   스케일러 -> 파일명 iris_scaler.pkl

   (위 저장 정보는 웹 애플리케이션에서 사용될것임)

⑪ 저장된 모델 & 스케일러 불러오기

⑫ 예측 함수 만들어 보고 동작 시키기

    # 예측 함수 작성
    # 입력값: 웹에서 사용자가 입력한 값
    # 출력값: 분류 문자열 (ex: Iris-setosa, Iris-versicolor, Iris-virginica)
    def predict_iris(sepal_length, sepal_width, petal_length, petal_width) -> str:


★ 각 수행 단계별로 적절한 제목으로 작성
★ 각 수행 단계마다 이 단계가 무엇을 하는 단계이고, 사용하는 파라미터와
   (필요한 경우) 어떠한 입력으로 어떠한 결과가 나오는지 확인하고 설명을 남기기
★ 각 단계마다 내가 무엇을 확인했는지 코드와 함께 설명 남기기
"""
None

# import

In [4]:
import os
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 데이터 불러오기

In [5]:
# base_path를 생성해 데이터 불러오기
base_path = r'F:\KDT2508\Dropbox\K01\PyWork\MachineLearning\proj_iris\model'
file_path = os.path.join(base_path, r'iris.csv')

In [6]:
df = pd.read_csv(file_path)
df.head() #불러온 데이터의 상의 5개 데이터 확인

,SepalLength,SepalWidth,PetalLength,PetalWidth,Name
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [7]:
df.info() # 누락된 데이터가 있지 않은지 확인 / 데이터 타입등 전반적인 데이터 확인

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   SepalLength  150 non-null    float64
 1   SepalWidth   150 non-null    float64
 2   PetalLength  150 non-null    float64
 3   PetalWidth   150 non-null    float64
 4   Name         150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB


In [8]:
df.describe() # scale이 필요한지, outlier는 없는지, 데이터가 well-balanced 인지 확인

,SepalLength,SepalWidth,PetalLength,PetalWidth
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.054000,3.758667,1.198667
std,0.828066,0.433594,1.764420,0.763161
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


# 데이터 분리

In [9]:
# 타켓데이터와 trian/test 데이터로 구분
df_data = ["SepalLength", "SepalWidth", "PetalLength", "PetalWidth"]
target = "Name"

X = df[df_data]
y = df[target]

# 전처리

In [10]:
scaler = StandardScaler()
scaler.fit(X)
X_scaled = scaler.transform(X)

X_scaled

array([[-9.00681170e-01,  1.03205722e+00, -1.34127240e+00,
        -1.31297673e+00],
       [-1.14301691e+00, -1.24957601e-01, -1.34127240e+00,
        -1.31297673e+00],
       [-1.38535265e+00,  3.37848329e-01, -1.39813811e+00,
        -1.31297673e+00],
       [-1.50652052e+00,  1.06445364e-01, -1.28440670e+00,
        -1.31297673e+00],
       [-1.02184904e+00,  1.26346019e+00, -1.34127240e+00,
        -1.31297673e+00],
       [-5.37177559e-01,  1.95766909e+00, -1.17067529e+00,
        -1.05003079e+00],
       [-1.50652052e+00,  8.00654259e-01, -1.34127240e+00,
        -1.18150376e+00],
       [-1.02184904e+00,  8.00654259e-01, -1.28440670e+00,
        -1.31297673e+00],
       [-1.74885626e+00, -3.56360566e-01, -1.34127240e+00,
        -1.31297673e+00],
       [-1.14301691e+00,  1.06445364e-01, -1.28440670e+00,
        -1.44444970e+00],
       [-5.37177559e-01,  1.49486315e+00, -1.28440670e+00,
        -1.31297673e+00],
       [-1.26418478e+00,  8.00654259e-01, -1.22754100e+00,
      

# 데이터 나누기

In [11]:
# scale 을 진행한 후에 데이터를 train/test 로 나눔 
X_train_scaled, X_test_scaled, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42, #결과의 일관성을 위해
    stratify=y # 비율이 균등하게 나눠지도록 유지함
)

In [13]:
y_train.value_counts()

Name
Iris-setosa        40
Iris-virginica     40
Iris-versicolor    40
Name: count, dtype: int64

In [ ]:
# 데이터가 잘 나누어 졌는지 확인
X_train_scaled.shape, X_test_scaled.shape

# 최적의 파라미터 찾기

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {'C': [0.1, 1, 10, 100],          
              'gamma': [0.001, 0.1, 1, 10, 100], 
              'kernel': ['linear', 'rbf']}

In [ ]:
grid = GridSearchCV(
    estimator=SVC(),
    param_grid=param_grid,
    cv=4,
)

grid.fit(X_train_scaled, y_train)

print("best_params_:", grid.best_params_)
print("best_cv_score:", grid.best_score_)

In [ ]:
clf = grid.best_estimator_
clf

# test 점수 확인

In [ ]:
y_pred = clf.predict(X_test_scaled)
test_acc = accuracy_score(y_test, y_pred)
print("test accuracy:", test_acc)

# 평가지표

In [ ]:
print(classification_report(y_test, y_pred))   # accuracy, precision, recall, f1-score 확인

# 예측 동작 확인

In [ ]:
# sameple data
sample_1 = [[5.1, 3.5, 1.4, 0.2]]
sample_2 = [[6.1, 3.1, 5.3, 2.2]]
sample_3 = [[6.9, 3.0, 4.5, 1.5]]

In [ ]:
# smaple data scaling
sample_scaled_1 = scaler.transform(sample_1)
sample_scaled_2 = scaler.transform(sample_2)
sample_scaled_3 = scaler.transform(sample_3)

In [ ]:
# 예측값 출력
print("예측 값 :", clf.predict(sample_scaled_1)[0])
print("예측 값 :", clf.predict(sample_scaled_2)[0])
print("예측 값 :", clf.predict(sample_scaled_3)[0])

# 파일저장

In [ ]:
import os, joblib

#base_dir 로 정확한 위치를 지정해서 파일 저장
base_dir = r"F:\KDT2508\Dropbox\K01\PyWork\MachineLearning\proj_iris\model"

joblib.dump(clf, os.path.join(base_dir, "iris_model.pkl")) 
joblib.dump(scaler, os.path.join(base_dir, "iris_scaler.pkl"))

# 파일 불러오기

In [ ]:
# 저장된 파일 확인
loaded_clf = joblib.load("iris_model.pkl")
loaded_scaler = joblib.load("iris_scaler.pkl")

print(type(loaded_clf), type(loaded_scaler))